In [9]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math

# List Of Tradeable Pairs And Indicators

In [10]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    "USDJPY"   # US Dollar / Japanese Yen
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Function to dynamically create variables for each pair
def create_dynamic_variables():
    for pair in pairs:
        # Get the indicators and price for the pair
        price, ema, rsi, atr = calculate_indicators(pair)

        if price is not None:
            # Get the latest Ask and Bid prices for the pair
            ask_price, bid_price = get_latest_prices(pair)

            # Create dynamic variables using globals() with the pair name and the data type
            globals()[f"{pair.lower()}_price"] = price
            globals()[f"{pair.lower()}_ema"] = ema
            globals()[f"{pair.lower()}_rsi"] = rsi
            globals()[f"{pair.lower()}_atr"] = atr
            globals()[f"{pair.lower()}_askprice"] = ask_price
            globals()[f"{pair.lower()}_bidprice"] = bid_price

# Run the function to create dynamic variables
create_dynamic_variables()

# Now, you can access the variables directly
print("Dynamic Variables for each pair:")
print(f"EURUSD Price: {eurusd_price}, EMA: {eurusd_ema}, RSI: {eurusd_rsi}, ATR: {eurusd_atr}, Ask Price: {eurusd_askprice}, Bid Price: {eurusd_bidprice}")
print(f"EURCHF Price: {eurchf_price}, EMA: {eurchf_ema}, RSI: {eurchf_rsi}, ATR: {eurchf_atr}, Ask Price: {eurchf_askprice}, Bid Price: {eurchf_bidprice}")
print(f"EURJPY Price: {eurjpy_price}, EMA: {eurjpy_ema}, RSI: {eurjpy_rsi}, ATR: {eurjpy_atr}, Ask Price: {eurjpy_askprice}, Bid Price: {eurjpy_bidprice}")
print(f"USDCHF Price: {usdchf_price}, EMA: {usdchf_ema}, RSI: {usdchf_rsi}, ATR: {usdchf_atr}, Ask Price: {usdchf_askprice}, Bid Price: {usdchf_bidprice}")
print(f"CHFJPY Price: {chfjpy_price}, EMA: {chfjpy_ema}, RSI: {chfjpy_rsi}, ATR: {chfjpy_atr}, Ask Price: {chfjpy_askprice}, Bid Price: {chfjpy_bidprice}")
print(f"USDJPY Price: {usdjpy_price}, EMA: {usdjpy_ema}, RSI: {usdjpy_rsi}, ATR: {usdjpy_atr}, Ask Price: {usdjpy_askprice}, Bid Price: {usdjpy_bidprice}")


Dynamic Variables for each pair:
EURUSD Price: 1.04898, EMA: 1.048665672776359, RSI: 43.99407925989127, ATR: 0.0005259634867399867, Ask Price: 1.04927, Bid Price: 1.04898
EURCHF Price: 0.94266, EMA: 0.9439470838493986, RSI: 36.61259759547583, ATR: 0.0004893972301054031, Ask Price: 0.94499, Bid Price: 0.94266
EURJPY Price: 159.74, EMA: 159.80436218552072, RSI: 43.72425855225825, ATR: 0.09556727219746637, Ask Price: 159.821, Bid Price: 159.74
USDCHF Price: 0.89916, EMA: 0.9001652113758754, RSI: 49.710556944600405, ATR: 0.0004618512190929923, Ask Price: 0.90013, Bid Price: 0.89916
CHFJPY Price: 169.061, EMA: 169.2588906944518, RSI: 32.72614721334256, ATR: 0.09905077832937223, Ask Price: 169.564, Bid Price: 169.061
USDJPY Price: 152.279, EMA: 152.39334707440133, RSI: 47.323486511461496, ATR: 0.08097045933286244, Ask Price: 152.327, Bid Price: 152.279


# DXY dollar index

In [11]:
def get_dxy_data():
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker("DX-Y.NYB")
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=16)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=64)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

dxy , dxy_ema, dxy_rsi = get_dxy_data()

print ('dxy: ' , dxy)
print ('dxy ema: ' , dxy_ema)
print ('dxy rsi: ' , dxy_rsi)

dxy:  106.79299926757812
dxy ema:  106.84881832022052
dxy rsi:  54.57480007906587


# EXY euro index

In [12]:
def get_exy_data():
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker("^XDE")
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=16)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=64)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

exy , exy_ema, exy_rsi = get_exy_data()

print ('exy: ' , exy)
print ('exy ema: ' , exy_ema)
print ('exy rsi: ' , exy_rsi)

exy:  104.91799926757812
exy ema:  104.57399140283533
exy rsi:  55.245134440071766


# XDS swiss index

In [13]:
def get_xds_data():
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker("^XDS")
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=16)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=64)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

xds , xds_ema, xds_rsi = get_xds_data()

print ('xds: ' , xds)
print ('xds ema: ' , xds_ema)
print ('xds rsi: ' , xds_rsi)

xds:  111.13580322265625
xds ema:  110.77164622338661
xds rsi:  52.16402836528199


# XDN japanese index

In [14]:
def get_xdn_data():
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker("^XDN")
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=16)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=64)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

xdn , xdn_ema, xdn_rsi = get_xdn_data()

print ('xdn: ' , xdn)
print ('xdn ema: ' , xdn_ema)
print ('xdn rsi: ' , xdn_rsi)

xdn:  65.64700317382812
xdn ema:  65.48794997890074
xdn rsi:  57.26524470847511


# Stop Loss And Take Profit Calculations

In [15]:
import math
account_info = mt5.account_info()
balance = account_info.balance
if balance:
    print(f"Account Balance: {balance}")
else:
    print("Could not retrieve account balance.")


# Function to calculate Stop Loss and Take Profit and store them as variables
def calculate_stop_loss_take_profit(pairs):
    # Convert the pair to lowercase (e.g., 'eurusd' instead of 'EURUSD')
    pair_key = pair.lower()

    # Get the ATR value from the corresponding variable
    if pair_key == 'eurusd':
        atr = eurusd_atr
    elif pair_key == 'eurchf':
        atr = eurchf_atr
    elif pair_key == 'eurjpy':
        atr = eurjpy_atr
    elif pair_key == 'usdchf':
        atr = usdchf_atr
    elif pair_key == 'chfjpy':
        atr = chfjpy_atr
    elif pair_key == 'usdjpy':
        atr = usdjpy_atr
    else:
        print(f"ATR not found for {pair}")
        return None, None

    # Calculate the Stop Loss (ATR * 2)
    stop_loss = atr * 2

    # Calculate the Take Profit (2 times the Stop Loss)
    take_profit = stop_loss * 2

    # Store the Stop Loss and Take Profit as individual variables
    globals()[f"{pair_key}_sl"] = stop_loss
    globals()[f"{pair_key}_tp"] = take_profit

    print(f"Stop Loss for {pair}: {stop_loss}")
    print(f"Take Profit for {pair}: {take_profit}")

# Loop through each pair in the list and calculate its Stop Loss and Take Profit
for pair in pairs:
    calculate_stop_loss_take_profit(pair)

# Print Stop Loss, Take Profit, and ATR for each pair
print(f"Stop Loss Price for EURUSD: {eurusd_price - eurusd_sl}, Take Profit Price for EURUSD: {eurusd_price + eurusd_tp}, ATR for EURUSD: {eurusd_atr}")
print(f"Stop Loss Price for EURCHF: {eurchf_price - eurchf_sl}, Take Profit Price for EURCHF: {eurchf_price + eurchf_tp}, ATR for EURCHF: {eurchf_atr}")
print(f"Stop Loss Price for EURJPY: {eurjpy_price - eurjpy_sl}, Take Profit Price for EURJPY: {eurjpy_price + eurjpy_tp}, ATR for EURJPY: {eurjpy_atr}")
print(f"Stop Loss Price for USDCHF: {usdchf_price - usdchf_sl}, Take Profit Price for USDCHF: {usdchf_price + usdchf_tp}, ATR for USDCHF: {usdchf_atr}")
print(f"Stop Loss Price for CHFJPY: {chfjpy_price - chfjpy_sl}, Take Profit Price for CHFJPY: {chfjpy_price + chfjpy_tp}, ATR for CHFJPY: {chfjpy_atr}")
print(f"Stop Loss Price for USDJPY: {usdjpy_price - usdjpy_sl}, Take Profit Price for USDJPY: {usdjpy_price + usdjpy_tp}, ATR for USDJPY: {usdjpy_atr}")


Account Balance: 160000.0
Stop Loss for EURUSD: 0.0010519269734799734
Take Profit for EURUSD: 0.002103853946959947
Stop Loss for EURCHF: 0.0009787944602108062
Take Profit for EURCHF: 0.0019575889204216123
Stop Loss for EURJPY: 0.19113454439493274
Take Profit for EURJPY: 0.3822690887898655
Stop Loss for USDCHF: 0.0009237024381859846
Take Profit for USDCHF: 0.001847404876371969
Stop Loss for CHFJPY: 0.19810155665874446
Take Profit for CHFJPY: 0.3962031133174889
Stop Loss for USDJPY: 0.16194091866572488
Take Profit for USDJPY: 0.32388183733144976
Stop Loss Price for EURUSD: 1.0479280730265201, Take Profit Price for EURUSD: 1.05108385394696, ATR for EURUSD: 0.0005259634867399867
Stop Loss Price for EURCHF: 0.9416812055397893, Take Profit Price for EURCHF: 0.9446175889204217, ATR for EURCHF: 0.0004893972301054031
Stop Loss Price for EURJPY: 159.54886545560507, Take Profit Price for EURJPY: 160.12226908878986, ATR for EURJPY: 0.09556727219746637
Stop Loss Price for USDCHF: 0.898236297561814,

In [21]:
def trade_strategy():
    
    sell_signal = False
    buy_signals = False
    for pair in pairs:
    
    print (latest_price)
    dxy , dxy_ema, dxy_rsi = get_dxy_data()
    exy , exy_ema, exy_rsi = get_exy_data()
    xds , xds_ema, xds_rsi = get_xds_data()
    xdn , xdn_ema, xdn_rsi = get_xdn_data()




IndentationError: expected an indented block after 'for' statement on line 5 (1535842094.py, line 6)